In [ ]:
# Run once to convert raw CSV to Parquet

import pandas as pd

cols = [
    "EPOCH_SECONDS", "CURRENCY_CODE", "USD_AMOUNT", "ORDERING_BIC8",
    "BENEFICIARY_BIC8", "BIC8_PATH", "TOTAL_DURATION", "CONTINUITY_QFLAG",
    "CURRENCY_CONVERSION_COUNT", "PARTICIPANT_COUNT", "TRANSACTION_TYPE_CODE",
    "ORDERING_TYPE", "BENEFICIARY_TYPE", "ORDERING_ACCOUNT", "BENEFICIARY_ACCOUNT",
    "ANOMALY_TYPE"  # Included so you can verify your rules against the labels!
]

df = pd.read_csv(
    "data/swift_umdac_datathon_2026_fictitious_payment_data.csv",
    usecols=cols
)

# Optional: Save your own local Parquet copy right away so future runs load in 2 seconds!
df.to_parquet("data/clean_data.parquet", index=False)

C:\Users\user\AppData\Local\Temp\ipykernel_10880\73269652.py:11: DtypeWarning: Columns (20: ANOMALY_TYPE) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(


In [3]:
df = pd.read_parquet("data/clean_data.parquet")
print(df.shape)
print(df["ANOMALY_TYPE"].value_counts(dropna=False))

(10000000, 16)
ANOMALY_TYPE
NaN                         9992994
ROMANCE_SCAM                   2476
FAT_FINGER_MISTAKE             1940
CEO_SCAM                        945
DUPLICATE_PAYMENT               679
ACCOUNT_REPLACEMENT_SCAM        649
CUSTOMER_RECALL                 317
Name: count, dtype: int64


In [ ]:
# Separate anomalies and normal baseline
anomalies = df[df["ANOMALY_TYPE"].notna()].copy()
normal_sample = df[df["ANOMALY_TYPE"].isna()].sample(n=50000, random_state=42).copy()

combined = pd.concat([anomalies, normal_sample], ignore_index=True)

# Keep ONLY assigned columns: party types, currencies, and routing behaviour
your_cols = [
    'ANOMALY_TYPE',
    # Party types
    'ORDERING_TYPE', 'BENEFICIARY_TYPE',
    # Currencies & FX
    'CURRENCY_CODE', 'USD_AMOUNT', 'CURRENCY_CONVERSION_COUNT',
    # Routing behaviour
    'ORDERING_BIC8', 'BENEFICIARY_BIC8', 'BIC8_PATH',
    'TOTAL_DURATION', 'CONTINUITY_QFLAG', 'PARTICIPANT_COUNT',
    'TRANSACTION_TYPE_CODE'
]

slice_df = combined[your_cols]
slice_df.to_csv("claude_routing_currency_party.csv", index=False)
print(f"Done! Saved {len(slice_df)} rows to claude_routing_currency_party.csv")

Done! Saved 57006 rows to claude_routing_currency_party.csv
